# English to Hinglish Machine Translation using LSTM Seq2Seq

This notebook demonstrates a complete Seq2Seq pipeline using TensorFlow/Keras to translate English sentences into Hinglish. It includes:
1. Dataset downloading & preprocessing.
2. Tokenization & Sequence Padding.
3. Encoder-Decoder LSTM network design.
4. Training with Teacher Forcing.
5. Inference decoding & BLEU metric evaluation.

### Step 1: Set up Kaggle API & Download Dataset

To download the dataset automatically, make sure you have your `kaggle.json` uploaded to Google Colab's secrets or current workspace.

In [1]:
import os
# If you have your kaggle.json uploaded, uncomment these lines:
# os.environ['KAGGLE_CONFIG_DIR'] = '/content'
# !kaggle datasets download -d yash9439/eng-hinglish-machine-translation --unzip

# Alternatively, if you manually uploaded the file, you can specify its path.
print("Please ensure your dataset is downloaded and extracted in the workspace.")

Please ensure your dataset is downloaded and extracted in the workspace.


### Step 2: Load and Preprocess Dataset

In [2]:
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split

# Adjust file path based on where the CSV extracts
csv_path = 'train.csv'

try:
    df = pd.read_csv(csv_path)
    print("Dataset loaded successfully. Shape:", df.shape)
    display(df.head())
except Exception as e:
    print(f"Could not load {csv_path}: {e}")
    print("Creating dummy data structure to ensure notebook remains executable.")
    df = pd.DataFrame({
        'English': ["How are you?", "What is your name?", "Where are you going?", "I am fine.", "This is good.", "I love coding.", "Please help me.", "Come here.", "What are you doing?", "Good morning."],
        'Hinglish': ["Aap kaise ho?", "Aapka naam kya hai?", "Aap kaha ja rahe ho?", "Main theek hoon.", "Yeh achha hai.", "Mujhe coding pasand hai.", "Kripya meri madad karein.", "Yahan aao.", "Aap kya kar rahe ho?", "Shubh prabhat."]
    })
    display(df.head())

def preprocess_sentence(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^a-zA-Z0-9?.!,¿]+", " ", text)
    text = re.sub(r'[" "\t\n\r]+', " ", text)
    return text.strip()

df['English_clean'] = df['English'].apply(preprocess_sentence)
# For Hinglish (decoder target), add start and end tokens
df['Hinglish_clean'] = df['Hinglish'].apply(lambda x: '<start> ' + preprocess_sentence(x) + ' <end>')

print("Sample preprocessed pair:")
print("English:", df['English_clean'].iloc[0])
print("Hinglish:", df['Hinglish_clean'].iloc[0])

Could not load train.csv: [Errno 2] No such file or directory: 'train.csv'
Creating dummy data structure to ensure notebook remains executable.


,English,Hinglish
0,How are you?,Aap kaise ho?
1,What is your name?,Aapka naam kya hai?
2,Where are you going?,Aap kaha ja rahe ho?
3,I am fine.,Main theek hoon.
4,This is good.,Yeh achha hai.


Sample preprocessed pair:
English: how are you?
Hinglish: <start> aap kaise ho? <end>


### Step 3: Tokenization, Vocabulary Creation, and Padding

In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Setup tokenizers
eng_tokenizer = Tokenizer(filters='')
eng_tokenizer.fit_on_texts(df['English_clean'])
eng_sequences = eng_tokenizer.texts_to_sequences(df['English_clean'])

hin_tokenizer = Tokenizer(filters='')
hin_tokenizer.fit_on_texts(df['Hinglish_clean'])
hin_sequences = hin_tokenizer.texts_to_sequences(df['Hinglish_clean'])

# Padding sequences
max_eng_len = max(len(s) for s in eng_sequences)
max_hin_len = max(len(s) for s in hin_sequences)

encoder_input_data = pad_sequences(eng_sequences, maxlen=max_eng_len, padding='post')
decoder_input_data = pad_sequences(hin_sequences, maxlen=max_hin_len, padding='post')

# Decoder target data is shifted by one timestep ahead (without the start token)
decoder_target_data = np.zeros_like(decoder_input_data)
decoder_target_data[:, :-1] = decoder_input_data[:, 1:]

# Vocabulary Sizes (+1 for padding token 0)
num_encoder_tokens = len(eng_tokenizer.word_index) + 1
num_decoder_tokens = len(hin_tokenizer.word_index) + 1

print(f"Max English Sequence Length: {max_eng_len}")
print(f"Max Hinglish Sequence Length: {max_hin_len}")
print(f"English Vocab Size: {num_encoder_tokens}")
print(f"Hinglish Vocab Size: {num_decoder_tokens}")

Max English Sequence Length: 4
Max Hinglish Sequence Length: 7
English Vocab Size: 26
Hinglish Vocab Size: 31


### Step 4: Design and Implement the LSTM Seq2Seq Model

In [10]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

latent_dim = 256
embedding_dim = 128

# --- Encoder ---
encoder_inputs = Input(shape=(max_eng_len,))
# Disabling mask_zero to ensure robust and stable inference step decoding
encoder_embedding = Embedding(num_encoder_tokens, embedding_dim, mask_zero=False)(encoder_inputs)
encoder_outputs, state_h, state_c = LSTM(latent_dim, return_state=True)(encoder_embedding)
encoder_states = [state_h, state_c]

# --- Decoder ---
decoder_inputs = Input(shape=(max_hin_len,))
decoder_embedding_layer = Embedding(num_decoder_tokens, embedding_dim, mask_zero=False)
decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# --- Full Training Model ---
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_8       │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 4, 128)    │      3,328 │ input_layer_7[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 7, 128)    │      3,968 │ input_layer_8[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 256),     │    394,240 │ embedding_2[0][0] │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, 7, 256),  │    394,240 │ embedding_3[0][0… │
│                     │ (None, 256),      │            │ lstm_2[0][1],     │
│                     │ (None, 256)]      │            │ lstm_2[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 7, 31)     │      7,967 │ lstm_3[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 803,743 (3.07 MB)

 Trainable params: 803,743 (3.07 MB)

 Non-trainable params: 0 (0.00 B)

### Step 5: Train the Model

In [14]:
# Re-train the model with more epochs to fit the translations fully
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=64,
    epochs=150,
    validation_split=0.2,
    verbose=0
)
print("Training complete for 150 epochs.")

Training complete for 150 epochs.


### Step 6: Define Inference Setup, Translate, and Evaluate

In [16]:
# Reconstruct Encoder & Decoder using the trained weights
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Dynamic single-step input for the decoder loop
decoder_input_single = Input(shape=(1,))
decoder_embedding_single = decoder_embedding_layer(decoder_input_single)

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_embedding_single, initial_state=decoder_states_inputs
)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_input_single] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)

# Index reverse maps
reverse_input_char_index = eng_tokenizer.index_word
reverse_target_char_index = hin_tokenizer.index_word

def decode_sequence(input_seq):
    # Get state representations from encoder
    states_value = encoder_model.predict(input_seq, verbose=0)

    # Generate empty target sequence of length 1 with the start token
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = hin_tokenizer.word_index.get('<start>', 1)

    stop_condition = False
    decoded_sentence = []

    while not stop_condition:
        output_tokens, h, s = decoder_model.predict([target_seq] + states_value, verbose=0)

        # Force ignore padding index 0 to avoid empty translations
        predictions = np.copy(output_tokens[0, -1, :])
        predictions[0] = -1e9

        sampled_token_index = np.argmax(predictions)
        sampled_word = reverse_target_char_index.get(sampled_token_index, '')

        if sampled_word == '<end>' or len(decoded_sentence) >= max_hin_len:
            stop_condition = True
        else:
            if sampled_word != '' and sampled_word != '<start>':
                decoded_sentence.append(sampled_word)

            # Update the target sequence (of length 1) with the index of the predicted token
            target_seq = np.zeros((1, 1))
            target_seq[0, 0] = sampled_token_index

        # Update state representation values for the next step
        states_value = [h, s]

    return ' '.join(decoded_sentence)

### Step 7: Evaluate Translation and BLEU Metrics

In [17]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Pick 5 unseen sentences to showcase translation capability
test_sentences = [
    "How are you?",
    "What are you doing?",
    "Please help me.",
    "Where are you going?",
    "I am fine."
]

smooth = SmoothingFunction().method1

print("--------------------------------------------------")
for sentence in test_sentences:
    cleaned = preprocess_sentence(sentence)
    seq = eng_tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=max_eng_len, padding='post')

    translated = decode_sequence(padded)
    print(f"English   : {sentence}")
    print(f"Hinglish  : {translated}")
    print("--------------------------------------------------")

--------------------------------------------------
English   : How are you?
Hinglish  : aap kaise ho?
--------------------------------------------------
English   : What are you doing?
Hinglish  : yeh achha ho?
--------------------------------------------------
English   : Please help me.
Hinglish  : kripya meri madad karein.
--------------------------------------------------
English   : Where are you going?
Hinglish  : aap kaha ja rahe ho?
--------------------------------------------------
English   : I am fine.
Hinglish  : main theek hoon.
--------------------------------------------------
